[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q1/05_gap_characterization.ipynb)

# R2-Q1 Week 4 — Characterize the PV → PD transfer gap

This is the last classroom notebook for R2-Q1. It picks up where
Notebook 04 left off.

Notebook 04 produced a headline number: your PV-trained classifier
loses some amount of accuracy when evaluated on PlantDoc rather than
PlantVillage, and the bootstrap confidence interval told you whether
that gap was real. What N04 did not tell you is *where* the gap lives
— which host species the model transfers cleanly to, which disease
categories it handles poorly, and which combinations drive most of
the failure.

This notebook decomposes the gap into two cuts:

1. **By host group.** Per-host accuracy across all disease categories.
   Answers: which plants does the model recognize cleanly when it
   sees them in a field photograph, and which does it stumble on?
2. **By disease category.** Per-category accuracy across all hosts.
   Answers: which disease categories transfer from lab to field, and
   which ones the model evidently learned in a PV-specific way?

By the end of this notebook you will have:

- A flat decomposition table (`gap_decomposition.parquet`) with one
  row per group: host or category name, sample count, accuracy, 95%
  bootstrap CI, and a flag for groups with too few samples to support
  a reliable confidence interval.
- A reader-facing summary (`characterization_summary.json`) naming
  the hosts and categories driving the gap and listing which cuts
  had enough data to be reliable.
- Two bar charts (`gap_by_host.png`, `gap_by_category.png`) with
  bootstrap error bars — the figures your Week 5 paper will lean on.

### What you need from N04

This notebook reads `pd_predictions.parquet` from your R2-Q1 output
directory (along with `precommit.json` for the disease-category
mapping). Both should already be there from running N04. Section 2
checks for them and fails with a clear error if either is missing.

### Install the iRI Lab package

If you opened this notebook in a fresh Colab session, you need to
install the package first. If you just finished running Notebook 04
in the same session, the package is already installed and you only
need to refresh it.

The diagnostic cell below tells you which of the two install patterns
in the following cell to uncomment. Run the diagnostic, read what it
prints, then uncomment exactly one line in the cell after it.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in Cell 1:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in Cell 1:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

In [ ]:
# Mount Drive, set up irilab2026 (no GPU required for this notebook),
# seed everything, and point OUTPUT_DIR at the R2-Q1 output directory.
import irilab2026 as iri

iri.setup(gpu_required=False)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q1")
print(f"Output directory: {OUTPUT_DIR}")

### Development mode

This notebook is fast — the per-group bootstrap is dataframe-only and
runs in seconds even at the precommit's full iteration count.

The `DEV_MODE` switch below still earns its keep on small iterations.
If you're tweaking a plot, fixing a typo, or debugging a save path,
clipping the bootstrap from a thousand iterations down to a hundred
shaves off most of the wait. The pattern matches the toggle you used
in Notebooks 03 and 04 — same shape, one knob fewer.

- **`DEV_MODE = True`**: cap the bootstrap at 100 iterations regardless
  of your precommit. The point estimates are still real, but the
  confidence intervals are wider than they would be at the precommit's
  full count. Numbers produced in dev mode are not paper-ready.
- **`DEV_MODE = False`**: honor your precommit's `n_bootstrap`. These
  are the numbers you report.

Ship the notebook with `DEV_MODE = False` so a student opening it
fresh produces real numbers. Flip to `True` only while you're
iterating on something.

In [ ]:
### 1.1) DEV_MODE switch

DEV_MODE = False

if DEV_MODE:
    BOOTSTRAP_CAP = 100      # max bootstrap iterations regardless of precommit
    print("⚠️  DEV_MODE is ON — numbers produced are NOT paper-ready.")
    print(f"    Bootstrap cap: {BOOTSTRAP_CAP} iterations")
else:
    BOOTSTRAP_CAP = None     # honor precommit's n_bootstrap
    print("DEV_MODE is OFF — full production run.")
    print(f"    Bootstrap cap: honor precommit n_bootstrap")

## 2) Read your inputs and pre-commits

This notebook reads two files produced by earlier work in R2-Q1:

- **`pd_predictions.parquet`** — N04's per-image predictions on
  PlantDoc's test split, with `host` and `true_category` metadata
  already joined in. Every row of analysis in this notebook starts
  from this table.
- **`precommit.json`** — your Week 1 methodology commitments. This
  notebook honors the bootstrap iteration count and significance
  level from there. It also reads `categories_used` so your bar
  charts order categories the same way the rest of R2-Q1 has.

If either file is missing from your R2-Q1 output directory, the
cells below fail with a message pointing you back to the notebook
that produces it. You can't run this notebook without both.

In [ ]:
### 2.1) Read precommit.json
import json

precommit_path = OUTPUT_DIR / "precommit.json"

if not precommit_path.exists():
    raise FileNotFoundError(
        f"Couldn't find {precommit_path}. "
        "Run Notebook 02 (`02_pd_orientation.ipynb`) to completion — "
        "the last section writes precommit.json to this location."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# Validate the three top-level keys exist. Each section below
# cherry-picks the specific fields it needs from these.
expected_keys = {"aggregation_level", "class_space_remapping", "statistical_test"}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing these top-level keys: {sorted(missing)}. "
        f"Re-run Notebook 02 to regenerate the file."
    )

print(f"Loaded: {precommit_path}")
print(f"  Top-level keys: {sorted(precommit.keys())}")

In [ ]:
### 2.2) Read pd_predictions.parquet
import pandas as pd

pd_predictions_path = OUTPUT_DIR / "pd_predictions.parquet"

if not pd_predictions_path.exists():
    raise FileNotFoundError(
        f"Couldn't find {pd_predictions_path}. "
        "Run Notebook 04 (`04_transfer_gap_measure.ipynb`) to completion — "
        "Section 5 writes pd_predictions.parquet to this location."
    )

pd_predictions = pd.read_parquet(pd_predictions_path)

# Validate the columns this notebook actually depends on.
expected_columns = {"host", "true_category", "correct_at_category"}
missing = expected_columns - set(pd_predictions.columns)
if missing:
    raise KeyError(
        f"pd_predictions.parquet is missing these columns: {sorted(missing)}. "
        f"Re-run Notebook 04 to regenerate the file."
    )

print(f"Loaded: {pd_predictions_path}")
print(f"  Shape: {pd_predictions.shape[0]} rows × {pd_predictions.shape[1]} columns")
print(f"  Columns: {list(pd_predictions.columns)}")

In [ ]:
### 2.3) Cherry-pick fields for downstream sections

# Sections 3 and 4 (host cut, category cut) use these.
n_bootstrap = precommit["statistical_test"]["n_bootstrap"]
alpha = precommit["statistical_test"]["alpha"]

# Section 5 (figures and summary) uses this for category ordering.
categories_used = precommit["class_space_remapping"]["categories_used"]

# Apply the DEV_MODE bootstrap cap if it's set.
if BOOTSTRAP_CAP is not None:
    if n_bootstrap > BOOTSTRAP_CAP:
        print(f"DEV_MODE: capping n_bootstrap from {n_bootstrap} to {BOOTSTRAP_CAP}")
        n_bootstrap = BOOTSTRAP_CAP

print("Pre-commits loaded:")
print(f"  Bootstrap iterations:  {n_bootstrap}")
print(f"  Significance level:    α = {alpha}")
print(f"  Categories committed:  {len(categories_used)} ({categories_used})")

Three notes on what these values commit you to and what they don't.

**The predictions table is the load-bearing input.** Every cut in
Sections 3 and 4 starts from `pd_predictions`. The host cut groups
on `host`; the category cut groups on `true_category`; both compute
accuracy from `correct_at_category`. If N04 ran with a different
precommit than the one loaded here — say, you re-ran N02 between
runs and the category mapping shifted — the predictions table
reflects N04's run, and this notebook's results follow the
predictions, not the precommit. Section 5's summary JSON captures
both so any divergence is visible.

**`categories_used` is informational with one operational use.** This
notebook reads it so the per-category bar chart can order its bars
consistently across runs. It's also written into the summary JSON
as methodological provenance. The analysis itself doesn't depend on
it — if `categories_used` and the categories present in the
predictions table disagree, the chart shows what's in the data.

**Everything else in `precommit.json` is informational here.** The
class-space mappings were already applied upstream in N04, so this
notebook doesn't re-apply them. The aggregation level and statistical
test choice are recorded into the summary JSON for provenance, but
this notebook ships with a fixed approach (percentile bootstrap CIs
on per-group accuracy) and doesn't branch on those fields.

## 3) Cut by host group

The first cut decomposes the PV → PD gap by host species. For each
host in PlantDoc — apple, tomato, corn, and so on — this section
computes how often the PV-trained classifier got the disease
category right when the image was of that host.

A host where the classifier is right 90% of the time tells you the
PV-learned features carried over cleanly to that host's field
photographs. A host where it's right 40% of the time tells you the
model didn't generalize to that host's appearance, even if it might
handle the host's diseases fine on PV-internal evaluation.

### How the bootstrap CI works here

For each host, we resample the host's predictions with replacement
`n_bootstrap` times (your precommit's value, or the DEV_MODE cap if
that's set lower). Each resample produces one accuracy number; the
95% interval comes from the 2.5th and 97.5th percentiles of that
distribution. The point estimate is the accuracy on the actual
(unsampled) data.

### The small-sample flag

PlantDoc has 236 test images spread across roughly thirteen hosts,
which leaves some hosts with fewer than 20 images. A bootstrap CI
on a sample that small is technically well-defined but so wide it
doesn't tell you much about the underlying accuracy.

This notebook flags any host with `n < 20` as `n_too_small = True`
in the decomposition table. The CI gets computed for those rows
anyway — recorded for transparency — but the bar charts in Section
5 will skip them, and your paper should treat their numbers as
suggestive at best.

The threshold of 20 is a notebook-author choice, not a value from
your precommit. It's a balance between "wide enough that the CI
means something" and "low enough that you don't lose too many
groups." If your dataset's host distribution is very different from
the expected one, the threshold may need rethinking — talk to your
mentor before changing it.

In [ ]:
### 3.1) Per-host sample distribution

per_host_n = (
    pd_predictions
    .groupby("host")
    .size()
    .sort_values(ascending=False)
    .rename("n")
    .to_frame()
)

print("Per-host sample counts:")
print(per_host_n.to_string())
print()
n_small = int((per_host_n["n"] < 20).sum())
n_ok = int((per_host_n["n"] >= 20).sum())
print(f"Hosts with n < 20:  {n_small}")
print(f"Hosts with n >= 20: {n_ok}")

In [ ]:
### 3.2) Compute per-host bootstrap CIs
import numpy as np

# Threshold for the small-sample flag. Notebook-author choice,
# not a precommit value. Section 4 duplicates this constant.
N_TOO_SMALL_THRESHOLD = 20

rng = np.random.default_rng(seed=42)

host_rows = []

for host_name, group_df in pd_predictions.groupby("host"):
    correct = group_df["correct_at_category"].to_numpy()
    n = len(correct)
    point_acc = correct.mean()

    # Pre-generate all resample indices in one (n_bootstrap, n) matrix,
    # then compute per-row accuracy in one vectorized pass.
    resample_idx = rng.integers(0, n, size=(n_bootstrap, n))
    boot_acc = correct[resample_idx].mean(axis=1)

    ci_lo, ci_hi = np.percentile(boot_acc, [100 * alpha / 2, 100 * (1 - alpha / 2)])

    host_rows.append({
        "cut_type":     "host",
        "group":        host_name,
        "n":            n,
        "accuracy":     point_acc,
        "ci_lo":        ci_lo,
        "ci_hi":        ci_hi,
        "n_too_small":  n < N_TOO_SMALL_THRESHOLD,
    })

host_decomp = pd.DataFrame(host_rows)
print(f"Computed bootstrap CIs for {len(host_decomp)} hosts.")
print(f"  Bootstrap iterations per host: {n_bootstrap}")
print(f"  Small-sample threshold:        n < {N_TOO_SMALL_THRESHOLD}")

In [ ]:
### 3.3) Show the results, sorted by accuracy

host_decomp_sorted = host_decomp.sort_values("accuracy", ascending=False)

print("Per-host accuracy with 95% bootstrap CIs:")
print()
print(host_decomp_sorted.to_string(index=False, float_format="%.3f"))
print()
n_reliable = int((~host_decomp_sorted["n_too_small"]).sum())
n_flagged = int(host_decomp_sorted["n_too_small"].sum())
print(f"Reliable rows (n >= {N_TOO_SMALL_THRESHOLD}):  {n_reliable}")
print(f"Flagged rows  (n <  {N_TOO_SMALL_THRESHOLD}):  {n_flagged}")

Three things to take from this table.

**The reliable rows tell you where transfer succeeded or failed.**
Hosts at the top of the sorted table — high accuracy, narrow CI —
are ones where the PV-learned features generalize cleanly. Hosts
near the bottom point to host appearances the model didn't pick up
in training. If your paper claims "the model transfers cleanly to
host X," cite that host's row and its CI.

**The flagged rows aren't useless, but treat them as suggestive.**
A host with 8 images and an accuracy of 0.25 might genuinely be a
case the model fails on, or it might be three unlucky samples. The
saved decomposition table records the number for transparency, but
your paper should be explicit about which findings rest on n ≥ 20
groups and which rest on smaller samples.

**The host cut alone doesn't tell the whole story.** A host might
show low accuracy because the model genuinely doesn't recognize
that plant, or because the disease categories present in that
host's images happen to be the ones the model handles poorly.
Section 4's category cut answers the second question directly.
Section 5 stacks both cuts into one decomposition table so the
joint picture is visible.

## 4) Cut by disease category

The second cut decomposes the gap by disease category. For each
category in your precommit's `categories_used` list — bacterial,
fungal, viral, healthy, or whatever your set is — this section
computes how often the PV-trained classifier got the category right
when the image was of that category.

A category where the classifier is right 90% of the time tells you
the model learned features that identify that disease type in PV
and that those features carried over to field photographs. A
category where it's right 30% of the time tells you the model's
PV-learned features for that disease don't survive the lab-to-field
shift — either the symptoms look meaningfully different in field
photos, or the model was relying on PV-specific shortcuts (uniform
backgrounds, controlled lighting) that PlantDoc images don't have.

The mechanics are identical to Section 3 — same bootstrap, same
threshold of 20, same `n_too_small` flag. The only difference is the
grouping column: `true_category` instead of `host`.

In [ ]:
### 4.1) Per-category sample distribution

per_category_n = (
    pd_predictions
    .groupby("true_category")
    .size()
    .sort_values(ascending=False)
    .rename("n")
    .to_frame()
)

print("Per-category sample counts:")
print(per_category_n.to_string())
print()
n_small = int((per_category_n["n"] < 20).sum())
n_ok = int((per_category_n["n"] >= 20).sum())
print(f"Categories with n < 20:  {n_small}")
print(f"Categories with n >= 20: {n_ok}")

In [ ]:
### 4.2) Compute per-category bootstrap CIs

# Threshold for the small-sample flag. Duplicated from Section 3
# intentionally — see Section 3's rationale.
N_TOO_SMALL_THRESHOLD = 20

rng = np.random.default_rng(seed=42)

category_rows = []

for category_name, group_df in pd_predictions.groupby("true_category"):
    correct = group_df["correct_at_category"].to_numpy()
    n = len(correct)
    point_acc = correct.mean()

    # Same vectorized bootstrap pattern as Section 3.2.
    resample_idx = rng.integers(0, n, size=(n_bootstrap, n))
    boot_acc = correct[resample_idx].mean(axis=1)

    ci_lo, ci_hi = np.percentile(boot_acc, [100 * alpha / 2, 100 * (1 - alpha / 2)])

    category_rows.append({
        "cut_type":     "category",
        "group":        category_name,
        "n":            n,
        "accuracy":     point_acc,
        "ci_lo":        ci_lo,
        "ci_hi":        ci_hi,
        "n_too_small":  n < N_TOO_SMALL_THRESHOLD,
    })

category_decomp = pd.DataFrame(category_rows)
print(f"Computed bootstrap CIs for {len(category_decomp)} categories.")
print(f"  Bootstrap iterations per category: {n_bootstrap}")
print(f"  Small-sample threshold:            n < {N_TOO_SMALL_THRESHOLD}")

In [ ]:
### 4.3) Show the results, sorted by accuracy

category_decomp_sorted = category_decomp.sort_values("accuracy", ascending=False)

print("Per-category accuracy with 95% bootstrap CIs:")
print()
print(category_decomp_sorted.to_string(index=False, float_format="%.3f"))
print()
n_reliable = int((~category_decomp_sorted["n_too_small"]).sum())
n_flagged = int(category_decomp_sorted["n_too_small"].sum())
print(f"Reliable rows (n >= {N_TOO_SMALL_THRESHOLD}):  {n_reliable}")
print(f"Flagged rows  (n <  {N_TOO_SMALL_THRESHOLD}):  {n_flagged}")

Two things to take from this table.

**The category cut answers the disease-side question.** Read together
with Section 3's host cut, you now have evidence for two distinct
claims: which hosts transferred and which didn't (Section 3), and
which disease categories transferred and which didn't (Section 4).
A finding that one host fails because its category is hard is
different from a finding that the same host fails even on a category
the model otherwise handles well. The two cuts together let you
distinguish those cases — though making the distinction precise
would need a cross-tabulation Section 5 deliberately doesn't build.

**Section 5 stacks the two decomposition tables and writes the
combined result to disk.** `host_decomp` and `category_decomp` will
become a single `gap_decomposition.parquet` with a `cut_type` column
distinguishing them. The two bar charts and the summary JSON come
from that combined table.

## 5) Synthesize and save

Sections 3 and 4 produced two in-memory dataframes — `host_decomp`
and `category_decomp` — with the same seven-column schema. This
section stacks them into one combined decomposition table, builds
the two bar charts, and writes a reader-facing summary JSON. Four
files end up on disk; they're what your Week 4 paper will draw from.

Subsection order:

1. **5.1) Stack and save the decomposition table.** Combines the
   two cuts into `gap_decomposition.parquet`.
2. **5.2 and 5.3) Build the bar charts.** One per cut, reliable
   rows only, alphabetical sort, fixed y-axis at `[0, 1]` so the
   two figures compare cleanly.
3. **5.4) Write the summary JSON.** Headline numbers for both
   cuts, methodology audit trail, one interpretation sentence
   built from the numeric variables.

In [ ]:
### 5.1) Stack the two decomposition tables and save

gap_decomposition = pd.concat([host_decomp, category_decomp], ignore_index=True)

decomp_path = OUTPUT_DIR / "gap_decomposition.parquet"
gap_decomposition.to_parquet(decomp_path, index=False)

# Round-trip verification — read it back and confirm it matches
roundtrip = pd.read_parquet(decomp_path)
pd.testing.assert_frame_equal(gap_decomposition, roundtrip)

print(f"Saved: {decomp_path}")
print(f"  Shape: {gap_decomposition.shape[0]} rows × {gap_decomposition.shape[1]} columns")
print(f"  Cuts:  {gap_decomposition['cut_type'].value_counts().to_dict()}")

In [ ]:
### 5.2) Build the per-host bar chart
import matplotlib.pyplot as plt

# Filter to reliable rows only — flagged rows are recorded in the
# saved decomposition table but don't go in the published figure.
host_reliable = host_decomp[~host_decomp["n_too_small"]].copy()

# Cosmetic: title-case the group names for display. PlantDoc has
# `grape` lowercase among otherwise TitleCase host names; this
# normalizes the x-axis labels without touching the saved data.
host_reliable["display_name"] = host_reliable["group"].str.title()

# Alphabetical sort — stable across runs, ordering-by-accuracy is
# already visible in Section 3.3's printed table.
host_reliable = host_reliable.sort_values("display_name")

# Error bars are distances from the point estimate, not absolute positions.
yerr_lower = host_reliable["accuracy"] - host_reliable["ci_lo"]
yerr_upper = host_reliable["ci_hi"] - host_reliable["accuracy"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    host_reliable["display_name"],
    host_reliable["accuracy"],
    yerr=[yerr_lower, yerr_upper],
    capsize=5,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy (category level)")
ax.set_xlabel("Host")
ax.set_title(
    f"Per-host accuracy of PV-trained classifier on PlantDoc\n"
    f"(reliable hosts only, n ≥ {N_TOO_SMALL_THRESHOLD}; "
    f"95% bootstrap CI, {n_bootstrap} iterations)"
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

host_fig_path = OUTPUT_DIR / "gap_by_host.png"
plt.savefig(host_fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {host_fig_path}")
print(f"  Plotted {len(host_reliable)} reliable host(s):")
for _, row in host_reliable.iterrows():
    print(f"    {row['display_name']:12s}  n = {row['n']:3d}  acc = {row['accuracy']:.3f}  CI = [{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]")

In [ ]:
### 5.3) Build the per-category bar chart

# Same pattern as 5.2 with one substitution: category_decomp instead
# of host_decomp.
category_reliable = category_decomp[~category_decomp["n_too_small"]].copy()
category_reliable["display_name"] = category_reliable["group"].str.title()
category_reliable = category_reliable.sort_values("display_name")

yerr_lower = category_reliable["accuracy"] - category_reliable["ci_lo"]
yerr_upper = category_reliable["ci_hi"] - category_reliable["accuracy"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    category_reliable["display_name"],
    category_reliable["accuracy"],
    yerr=[yerr_lower, yerr_upper],
    capsize=5,
    edgecolor="black",
    linewidth=0.5,
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy (category level)")
ax.set_xlabel("Disease category")
ax.set_title(
    f"Per-category accuracy of PV-trained classifier on PlantDoc\n"
    f"(reliable categories only, n ≥ {N_TOO_SMALL_THRESHOLD}; "
    f"95% bootstrap CI, {n_bootstrap} iterations)"
)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

category_fig_path = OUTPUT_DIR / "gap_by_category.png"
plt.savefig(category_fig_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved: {category_fig_path}")
print(f"  Plotted {len(category_reliable)} reliable categor(ies):")
for _, row in category_reliable.iterrows():
    print(f"    {row['display_name']:12s}  n = {row['n']:3d}  acc = {row['accuracy']:.3f}  CI = [{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]")

In [ ]:
### 5.4) Synthesize characterization_summary.json

# Pull two precommit fields not previously hoisted into named variables —
# they're recorded here for the methodology audit but not branched on.
aggregation_choice = precommit["aggregation_level"]["choice"]
test_choice = precommit["statistical_test"]["choice"]
n_bootstrap_committed = precommit["statistical_test"]["n_bootstrap"]

# Best / worst within each cut, restricted to reliable rows.
def best_and_worst(decomp_df):
    reliable = decomp_df[~decomp_df["n_too_small"]]
    best_row = reliable.loc[reliable["accuracy"].idxmax()]
    worst_row = reliable.loc[reliable["accuracy"].idxmin()]

    def row_to_dict(row):
        return {
            "group":    str(row["group"]),
            "n":        int(row["n"]),
            "accuracy": float(row["accuracy"]),
            "ci_lo":    float(row["ci_lo"]),
            "ci_hi":    float(row["ci_hi"]),
        }

    return row_to_dict(best_row), row_to_dict(worst_row)

best_host, worst_host = best_and_worst(host_decomp)
best_category, worst_category = best_and_worst(category_decomp)

# Build the interpretation sentence from numeric variables (refactor-safe —
# JSON refactors can't break the sentence because it's built from the
# same Python variables the JSON gets built from).
interpretation = (
    f"Among hosts with at least {N_TOO_SMALL_THRESHOLD} test images, "
    f"PV → PD accuracy ranges from "
    f"{best_host['accuracy']:.1%} ({best_host['group']}) to "
    f"{worst_host['accuracy']:.1%} ({worst_host['group']}). "
    f"Among categories meeting the same threshold, "
    f"{best_category['group']} transfers at {best_category['accuracy']:.1%} "
    f"(95% CI [{best_category['ci_lo']:.1%}, {best_category['ci_hi']:.1%}]) "
    f"while {worst_category['group']} drops to {worst_category['accuracy']:.1%} "
    f"(95% CI [{worst_category['ci_lo']:.1%}, {worst_category['ci_hi']:.1%}])."
)

# Methodology audit: record committed vs implemented; the only deviation
# the notebook can detect mechanically is the DEV_MODE bootstrap cap.
deviations = []
if n_bootstrap != n_bootstrap_committed:
    deviations.append(
        f"n_bootstrap was capped from {n_bootstrap_committed} to {n_bootstrap} (DEV_MODE)"
    )

# Reliable / flagged counts per cut.
host_reliable_count = int((~host_decomp["n_too_small"]).sum())
host_flagged_count = int(host_decomp["n_too_small"].sum())
category_reliable_count = int((~category_decomp["n_too_small"]).sum())
category_flagged_count = int(category_decomp["n_too_small"].sum())

characterization_summary = {
    "results": {
        "host_cut": {
            "reliable_count": host_reliable_count,
            "flagged_count":  host_flagged_count,
            "best":           best_host,
            "worst":          worst_host,
        },
        "category_cut": {
            "reliable_count": category_reliable_count,
            "flagged_count":  category_flagged_count,
            "best":           best_category,
            "worst":          worst_category,
        },
    },
    "methodology": {
        "committed": {
            "n_bootstrap":        n_bootstrap_committed,
            "alpha":              alpha,
            "aggregation_choice": aggregation_choice,
            "test_choice":        test_choice,
            "categories_used":    list(categories_used),
        },
        "implemented": {
            "n_bootstrap":            n_bootstrap,
            "alpha":                  alpha,
            "aggregation":            "disease_category",
            "test":                   "percentile_bootstrap",
            "small_sample_threshold": N_TOO_SMALL_THRESHOLD,
            "approach":               "Per-group percentile bootstrap CIs on category-level accuracy; one-dimensional cuts (host and category) only.",
        },
        "deviations_from_precommit": deviations,
        "dev_mode": DEV_MODE,
    },
    "interpretation": interpretation,
}

summary_path = OUTPUT_DIR / "characterization_summary.json"
with open(summary_path, "w") as f:
    json.dump(characterization_summary, f, indent=2)

print(f"Saved: {summary_path}")
print(f"  Size: {summary_path.stat().st_size} bytes")
print()
print("Headline interpretation:")
print(f"  {interpretation}")

Four files are now in your R2-Q1 output directory:

| File | Contents |
|---|---|
| `gap_decomposition.parquet` | One row per group across both cuts. Columns: `cut_type`, `group`, `n`, `accuracy`, `ci_lo`, `ci_hi`, `n_too_small`. Includes both reliable and flagged rows. |
| `characterization_summary.json` | Reader-facing finding: best and worst reliable group per cut, methodology audit trail, one headline interpretation sentence. |
| `gap_by_host.png` | Bar chart of per-host accuracy (reliable hosts only) with 95% bootstrap error bars. |
| `gap_by_category.png` | Bar chart of per-category accuracy (reliable categories only) with 95% bootstrap error bars. |

These four files together with the four files N04 produced
(`pv_predictions.parquet`, `pd_predictions.parquet`,
`transfer_gap.parquet`, `gap_summary.json`) are the full evidence base
for your Week 4 paper. Section 6 covers what to do with them.